# Tests: `fastermodels.model` (source `nbs/00_model.ipynb`)

In [ ]:
from fastcore.test import *
import json, shutil, tempfile
from pathlib import Path

import torch
import torch.nn as nn
import torch_pruning as tp
from safetensors.torch import load_file, save_file
from torchvision.models import mobilenet_v3_large, resnet18

from fastermodels.model import FasterModel, LAYER_TYPES, resolve, spec_from, state_hash

In [ ]:
def pruned_resnet18():
    "A resnet18 pruned non-uniformly: 20 % of a residual group (which couples conv1) and 50 % of layer3.0.conv1"
    torch.manual_seed(0)
    m = resnet18(num_classes=10, weights=None)
    dg = tp.DependencyGraph().build_dependency(m, example_inputs=torch.randn(1, 3, 64, 64))
    for layer, fraction in [(m.layer1[0].conv2, .2), (m.layer3[0].conv1, .5)]:
        idxs = list(range(int(layer.out_channels * fraction)))
        dg.get_pruning_group(layer, tp.prune_conv_out_channels, idxs=idxs).prune()
    return m.eval()


_pruned = pruned_resnet18()
_x = torch.randn(2, 3, 64, 64)

# the prune changed two groups, and conv1 came with the residual one
test_eq(_pruned.conv1.out_channels, 52)
test_eq(_pruned.layer3[0].conv1.out_channels, 128)

In [ ]:
# Round trip: wrap an optimized model, save it, load it back
_fm = FasterModel.wrap(_pruned, 'torchvision.models.resnet18', {'num_classes': 10, 'weights': None},
                       recipe={'prune': '20 % of a residual group, 50 % of layer3.0.conv1'})
test_eq(_fm.training, False)   # eval mode, or every forward reads and moves the BatchNorm statistics

with tempfile.TemporaryDirectory() as _d:
    _fm.save_pretrained(_d)
    _cfg = json.loads(Path(_d, 'config.json').read_text())
    # the spec travels with the weights: the coupled conv1 and the BatchNorm arguments
    test_eq(_cfg['modules']['conv1']['out_channels'], 52)
    test_eq(_cfg['modules']['conv1']['type'], 'Conv2d')
    test_eq(_cfg['modules']['bn1']['num_features'], 52)
    test_eq(_cfg['modules']['bn1']['eps'], 1e-5)
    test_eq(_cfg['source'], 'torchvision.models.resnet18')
    test_eq(_cfg['source_kwargs'], {'num_classes': 10, 'weights': None})

    _back = FasterModel.from_pretrained(_d)
    test_eq(_back.training, False)
    test_eq(state_hash(_back), state_hash(_fm))
    assert torch.equal(_back(_x), _fm(_x))
    assert torch.equal(_fm(_x), _pruned(_x))
    test_eq(spec_from(_back.net), spec_from(_pruned))

    # a batch whose statistics are not the published ones: the reload answers like the source anyway
    _x4 = torch.randn(4, 3, 64, 64)
    assert torch.equal(_pruned.eval()(_x4), _back(_x4))
    test_eq(state_hash(_back), state_hash(_fm))   # and those forwards moved no running statistic

In [ ]:
# Loading is strict: a spec that disagrees with the weights raises, it never loads in part
with tempfile.TemporaryDirectory() as _d:
    _fm.save_pretrained(_d)
    _bad = Path(_d, 'tampered')
    shutil.copytree(_d, _bad)

    _cfg = json.loads(Path(_bad, 'config.json').read_text())
    _cfg['modules']['conv1']['out_channels'] -= 1
    Path(_bad, 'config.json').write_text(json.dumps(_cfg))
    with ExceptionExpected(RuntimeError): FasterModel.from_pretrained(str(_bad))

    Path(_bad, 'config.json').write_text(Path(_d, 'config.json').read_text())
    _sd = load_file(Path(_d, 'model.safetensors'))
    del _sd['net.fc.bias']
    save_file(_sd, str(Path(_bad, 'model.safetensors')))
    with ExceptionExpected(RuntimeError): FasterModel.from_pretrained(str(_bad))

In [ ]:
# resolve names what it could not import
test_eq(resolve('torchvision.models.resnet18'), resnet18)
with ExceptionExpected(ImportError, regex='nope.nothing'): resolve('nope.nothing')
with ExceptionExpected(ImportError, regex='torchvision.models.not_a_model'): resolve('torchvision.models.not_a_model')

# a spec entry that names no module in the source model names itself in the error
with ExceptionExpected(KeyError, regex='does.not.exist'):
    FasterModel('torchvision.models.resnet18', {'num_classes': 10, 'weights': None},
                {'does.not.exist': {'type': 'Linear', 'in_features': 8, 'out_features': 4, 'bias': True}})

# an empty spec rebuilds the stock architecture
test_eq(FasterModel('torchvision.models.resnet18', {'num_classes': 10, 'weights': None}).net.conv1.out_channels, 64)

In [ ]:
# The spec is read off the live model, so it catches what a fixed list would miss
_spec = spec_from(mobilenet_v3_large(weights=None))
test_eq(_spec['features.1.block.0.0']['groups'], 16)          # depthwise convolution
test_eq(_spec['features.1.block.0.0']['type'], 'Conv2d')
test_close(_spec['features.0.1']['eps'], 0.001, eps=1e-9)     # BatchNorm arguments torchvision changed
test_close(_spec['features.0.1']['momentum'], 0.01, eps=1e-9)
test_eq(_spec['classifier.0']['type'], 'Linear')
assert set(_spec) and all(v['type'] in LAYER_TYPES for v in _spec.values())

# every value is JSON-native, so it survives config.json unchanged
test_eq(json.loads(json.dumps(_spec)), _spec)
test_eq(_spec['features.0.0']['kernel_size'], [3, 3])

# state_hash reads a model or a state dict, and a single changed weight changes it
_m = nn.Linear(4, 3)
test_eq(state_hash(_m), state_hash(_m.state_dict()))
_before = state_hash(_m)
with torch.no_grad(): _m.bias[0] += 1e-6
test_ne(state_hash(_m), _before)